Step 1: Define File Path and Load Data
This step defines the file path for the dataset and checks if it exists. If the file is found, it loads the dataset while ensuring column names are clean by removing any whitespace. This setup ensures a smooth loading process and prevents errors from missing files.

In [1]:
# Import necessary libraries for data manipulation, geospatial analysis, and plotting
import os
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt


In [2]:
# Define the file path to the dataset for loading
file_path = r'C:\Users\sul19\Desktop\701 Project\Property Assessment Datasets\Property Assessment 2016 V1.xlsx'

# Check if the file exists at the given path before loading it to handle errors gracefully
if os.path.exists(file_path):
    print("File found! Proceeding to load the data.")
    
    # Load the Excel dataset using 'openpyxl' engine to manage cell formatting
    df = pd.read_excel(file_path, engine='openpyxl', na_values=['', ' '])

    # Clean column names by removing any whitespace
    df.columns = df.columns.str.strip()
else:
    print(f"File not found at {file_path}. Please check the file path.")


File found! Proceeding to load the data.


Step 2: Clean ZIP_CODE Column and Standardize OVERALL_COND Column
This step standardizes the ZIP_CODE column by removing leading zeros and underscores, ensuring consistency in ZIP code format. It also maps values in the OVERALL_COND column to readable labels and removes invisible characters or empty values to enhance data quality for analysis.

In [3]:
# Remove leading zeros and trailing underscores from ZIP codes for standardization
df['ZIP_CODE'] = df['ZIP_CODE'].astype(str).str.lstrip('0').str.rstrip('_')

# Standardize values in 'OVERALL_COND' to create uniform categories for analysis
df['OVERALL_COND'] = df['OVERALL_COND'].replace({
    'A': 'A - Average',
    'G': 'G - Good',
    'E': 'E - Excellent',
    'F': 'F - Fair',
    'P': 'P - Poor'
})

# Remove any invisible characters or spaces in 'OVERALL_COND' to ensure data consistency
df['OVERALL_COND'] = df['OVERALL_COND'].astype(str).str.replace('\u00A0', '').str.strip()

# Replace any empty cells or whitespace-only values with 'none' for easier handling later
df['OVERALL_COND'] = df['OVERALL_COND'].replace(r'^\s*$', 'none', regex=True)

# Further condition replacements to account for known inconsistencies
df['OVERALL_COND'] = df['OVERALL_COND'].replace({
    'AVG - Default - Average': 'A - Average',
    'EX - Excellent': 'E - Excellent'
}, regex=False)

Step 3: Group Data by ZIP_CODE and Summarize OVERALL_COND Counts
In this step, we summarize housing conditions by counting occurrences of each condition within each ZIP code. This provides a general overview of housing conditions and helps identify areas with higher or lower ratings.

In [4]:
# Summarize housing conditions by ZIP code to get an overview of each area
overall_cond_summary = df.groupby('ZIP_CODE')['OVERALL_COND'].value_counts().unstack().fillna(0)

# Display the summary of housing conditions by ZIP code for analysis
print("Housing condition summary by ZIP code:")
print(overall_cond_summary)


Housing condition summary by ZIP code:
OVERALL_COND  A - Average  E - Excellent  F - Fair  G - Good  P - Poor     nan
ZIP_CODE                                                                      
2090                  0.0            0.0       0.0       0.0       0.0     1.0
2108                 40.0           76.0       3.0     137.0       1.0  1835.0
2109                 18.0            1.0       1.0       3.0       0.0  1835.0
2110                  0.0            0.0       0.0       0.0       0.0  1707.0
2111                 14.0            0.0       8.0       1.0       0.0  2497.0
2112                  0.0            0.0       0.0       0.0       0.0     1.0
2113                 88.0            1.0      14.0      28.0       1.0  2069.0
2114                 91.0           22.0      10.0     132.0       2.0  4562.0
2115                 88.0            3.0       2.0      79.0       0.0  4815.0
2116                178.0           89.0      20.0     201.0       7.0  8751.0
2118         

Step 4: Map Condition Labels to Numeric Values for Analysis
To enable quantitative analysis, this step converts condition labels into numeric scores. We then calculate the mean condition score for each ZIP code, offering insights into the average housing conditions by area.

Step 5: Convert Mean Condition Scores to Descriptive Labels
This step maps numeric mean scores back to descriptive labels to make results easier to interpret. Each ZIP code receives a label that represents the general housing condition.

In [5]:
# Define a mapping of housing conditions to numeric scores for quantitative analysis
def condition_to_numeric(cond):
    mapping = {
        'E - Excellent': 5,
        'VG - Very Good': 4,
        'G - Good': 3.5,
        'A - Average': 3,
        'F - Fair': 2,
        'P - Poor': 1.5,
        'VP - Very Poor': 1,
        'US - Unsound': 0,
        'none': np.nan  # Treat 'none' as NaN for numerical purposes
    }
    return mapping.get(cond, np.nan)

# Apply the numeric mapping to create a new column for condition scores
df['OVERALL_COND_NUM'] = df['OVERALL_COND'].apply(condition_to_numeric)

# Calculate the mean condition score by ZIP code to evaluate housing condition patterns
condition_analysis = df.groupby('ZIP_CODE').agg(
    overall_cond_mean=('OVERALL_COND_NUM', 'mean'),
).reset_index()

# Display condition analysis based on ZIP codes
print("Housing condition analysis by ZIP code:")
print(condition_analysis)

# Define a function to map mean condition scores back to descriptive labels
def mean_to_condition(mean_score):
    if mean_score >= 4.75:
        return 'Excellent'
    elif mean_score >= 4:
        return 'Very Good'
    elif mean_score >= 3.5:
        return 'Good'
    elif mean_score >= 3:
        return 'Average'
    elif mean_score >= 2:
        return 'Fair'
    elif mean_score >= 1:
        return 'Poor'
    elif mean_score >= 0.5:
        return 'Very Poor'
    else:
        return 'Unsound'

# Apply mapping to add descriptive labels to ZIP code conditions
condition_analysis['overall_cond_label'] = condition_analysis['overall_cond_mean'].apply(mean_to_condition)

# Print condition analysis with descriptive labels by ZIP code
print("Overall Condition Analysis by ZIP Code:")
print(condition_analysis[['ZIP_CODE', 'overall_cond_mean', 'overall_cond_label']])


Housing condition analysis by ZIP code:
   ZIP_CODE  overall_cond_mean
0      2090                NaN
1      2108           3.840467
2      2109           3.108696
3      2110                NaN
4      2111           2.673913
5      2112                NaN
6      2113           3.003788
7      2114           3.377432
8      2115           3.252907
9      2116           3.501010
10     2118           3.335253
11     2119           3.070434
12     2120           3.086957
13     2121           3.052731
14     2122           3.034542
15     2124           3.055287
16     2125           3.053840
17     2126           3.033871
18     2127           3.085882
19     2128           3.040954
20     2129           3.161675
21     2130           3.128672
22     2131           3.043828
23     2132           3.061912
24     2133                NaN
25     2134           3.025020
26     2135           3.032005
27     2136           3.035693
28     2137                NaN
29     2186           3.500000

Step 6: Standardize YR_BUILT and YR_REMODEL Columns
In this step, we ensure that year columns are numeric and remove any values beyond 2024, which would represent invalid future dates. This standardization allows us to calculate the mean construction and remodel years by ZIP code for further analysis.

Step 7: Classify Buildings by Age
This step categorizes buildings as 'Old,' 'Average,' or 'New' based on their construction or remodel year, adding insights into the age distribution within each ZIP code.

In [6]:
# Ensure YR_BUILT and YR_REMODEL are numeric and replace years > 2024 with NaN
df['YR_BUILT'] = pd.to_numeric(df['YR_BUILT'], errors='coerce')
df['YR_REMODEL'] = pd.to_numeric(df['YR_REMODEL'], errors='coerce')
df['YR_BUILT'] = df['YR_BUILT'].apply(lambda x: x if x <= 2024 else np.nan)
df['YR_REMODEL'] = df['YR_REMODEL'].apply(lambda x: x if x <= 2024 else np.nan)

# Group by ZIP_CODE and calculate the mean for YR_BUILT and YR_REMODEL
condition_analysis = df.groupby('ZIP_CODE').agg(
    yr_built_mean=('YR_BUILT', 'mean'),
    yr_remodel_mean=('YR_REMODEL', 'mean')
).reset_index()

# Define thresholds for building classification
def classify_building(yr_built, yr_remodel, old_threshold=1970, new_threshold=2000):
    """
    Classify building as 'Old', 'Average', or 'New' based on YR_REMODEL or YR_BUILT.
    If YR_REMODEL exists, use it; otherwise, use YR_BUILT.
    """
    if not pd.isna(yr_remodel):
        year = yr_remodel  # Use YR_REMODEL if available
    else:
        year = yr_built  # Otherwise, use YR_BUILT
    
    if pd.isna(year):
        return 'Unknown'
    elif year <= old_threshold:
        return 'Old'
    elif year >= new_threshold:
        return 'New'
    else:
        return 'Average'
    
# Apply classification based on YR_BUILT and YR_REMODEL
condition_analysis['building_classification'] = condition_analysis.apply(
    lambda row: classify_building(row['yr_built_mean'], row['yr_remodel_mean']), axis=1)

# Print the classification results based on the YR_BUILT and YR_REMODEL means
print("Building Classification Based on YR_BUILT and YR_REMODEL:")
print(condition_analysis[['ZIP_CODE', 'yr_built_mean', 'yr_remodel_mean', 'building_classification']])

# Define a function to map mean condition scores back to descriptive labels
def mean_to_condition(mean_score):
    if mean_score >= 4.75:
        return 'Excellent'
    elif mean_score >= 4:
        return 'Very Good'
    elif mean_score >= 3.5:
        return 'Good'
    elif mean_score >= 3:
        return 'Average'
    elif mean_score >= 2:
        return 'Fair'
    elif mean_score >= 1:
        return 'Poor'
    elif mean_score >= 0.5:
        return 'Very Poor'
    else:
        return 'Unsound'

# Apply mapping to add descriptive labels to ZIP code conditions
condition_analysis['overall_cond_label'] = condition_analysis['overall_cond_mean'].apply(mean_to_condition)

# Print condition analysis with descriptive labels by ZIP code
print("Overall Condition Analysis by ZIP Code:")
print(condition_analysis[['ZIP_CODE', 'overall_cond_mean', 'overall_cond_label']])


Building Classification Based on YR_BUILT and YR_REMODEL:
   ZIP_CODE  yr_built_mean  yr_remodel_mean building_classification
0      2090    1988.000000      1988.000000                 Average
1      2108    1545.926923      1890.452918                     Old
2      2109    1746.417121      1576.562584                     Old
3      2110    1506.829408      1006.930972                     Old
4      2111    1712.297471      1847.288633                     Old
5      2112       0.000000              NaN                     Old
6      2113    1812.134431      1891.280144                     Old
7      2114    1657.584940      1824.472308                     Old
8      2115    1653.872724      1862.459108                     Old
9      2116    1816.138689      1840.398909                     Old
10     2118    1787.740217      1888.894055                     Old
11     2119    1441.053592      1217.785935                     Old
12     2120    1578.753302      1194.362922               

Step 8: Include Address Details and Finalize Output
In this final step, we merge street address details into the main DataFrame for a more comprehensive summary that includes ZIP code, condition, and address information. We then save the output to an Excel file for further use.

In [7]:
# Assuming 'ST_NUM' and 'ST_NAME' are part of the original dataset
# Extract those columns from the original dataset
st_num_name = df[['ZIP_CODE', 'ST_NUM', 'ST_NAME']].drop_duplicates()

# Merge 'st_num_name' with 'condition_analysis' to include 'ST_NUM' and 'ST_NAME' with the results
condition_analysis = pd.merge(condition_analysis, st_num_name, on='ZIP_CODE', how='left')

# Add overall condition label based on previous analysis
# Ensure that the column 'overall_cond_label' from earlier condition analysis is merged correctly
overall_cond_summary = df.groupby('ZIP_CODE').agg(
    overall_cond_mean=('OVERALL_COND_NUM', 'mean'),
).reset_index()

# Reapply the condition label mapping function to map numeric values to condition labels
def mean_to_condition(mean_score):
    if mean_score >= 4.75:
        return 'Excellent'
    elif mean_score >= 4:
        return 'Very Good'
    elif mean_score >= 3.5:
        return 'Good'
    elif mean_score >= 3:
        return 'Average'
    elif mean_score >= 2:
        return 'Fair'
    elif mean_score >= 1:
        return 'Poor'
    elif mean_score >= 0.5:
        return 'Very Poor'
    else:
        return 'Unsound'

overall_cond_summary['overall_cond_label'] = overall_cond_summary['overall_cond_mean'].apply(mean_to_condition)

# Merge overall condition labels with the condition_analysis DataFrame
condition_analysis = pd.merge(condition_analysis, overall_cond_summary[['ZIP_CODE', 'overall_cond_label']], on='ZIP_CODE', how='left')

# Assign a constant value for 'YEAR'
condition_analysis['YEAR'] = 2016

# Rearranging the columns as requested
final_output = condition_analysis[['YEAR', 'ST_NUM', 'ST_NAME', 'ZIP_CODE', 'overall_cond_label', 'building_classification']]

# Save the final result to a new Excel file
output_file_path = 'Property_Assessment_2016_Output.xlsx'
final_output.to_excel(output_file_path, index=False)

print(f"File saved successfully to {output_file_path}")

File saved successfully to Property_Assessment_2016_Output.xlsx
